# S06: two applications, one shared platform

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/05_shared_platform.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

70-minute lab. Prove reuse with two applications that have different contracts. A platform provides controlled shared capabilities; a single chatbot renamed a platform does not establish reuse.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–20 min | Onboard and exercise both applications.** Policy serves buyers; spend requires finance. Shared admission control, model adapter, index version, traces and quotas live in `Platform`. The service determines identity and application. Read `engine/platform.py` before changing it.

In [ ]:
ix=Index();ix.build('v1',read_data('policies.json'));db=Purchases()
platform=Platform(ix,db,client,quota=3)
print(platform.ask(BUYER,'policy','standard supplier payment terms'))
assert platform.ask(BUYER,'spend','Nova Q1 spend')['status']=='denied'
print(platform.ask(FINANCE,'spend','Nova spend in 2026-Q1?',planned_query={'quarter':'2026-Q1','supplier':'Nova'}))

**20–40 min | Tenant, cache and quota boundaries.** The same question has different policy answers in AsterWorks and Beacon. Run repeated calls and inspect cache hits. A quota is scoped by tenant, subject and application; this is a process-local call allowance, not a distributed token or spend limit.

In [ ]:
q='standard supplier payment terms'
a=platform.ask(BUYER,'policy',q);b=platform.ask(BEACON,'policy',q)
print('Aster:',a);print('Beacon:',b)
assert not b.get('cache',False)
platform.ask(BUYER,'policy',q)
assert platform.ask(BUYER,'policy',q)['status']=='quota'
assert platform.ask(BEACON,'policy',q)['status']!='quota'
print(json.dumps(platform.events,indent=2))

**40–60 min | Make a platform change.** Add per-application quota configuration with default-preserving behavior. Write an assertion proving policy traffic cannot consume the spend allowance. Add a new entitlement version and demonstrate it does not reuse the old cache entry. Decide how a real entitlement change would invalidate an already-issued identity/session. Do not accept tenant or role from a model reply.

**60–70 min | Deliver an onboarding contract.** Specify owner, task contract, model permissions, data scope, request/token budget, SLO, audit fields and release owner for a proposed third application. Identify what belongs in shared infrastructure and what remains application logic. A single-process lab does not implement enterprise IAM, a distributed gateway or distributed quotas.

In [ ]:
ix.close();db.close()